# omicsTL — Transfer Learning for Omics Data

## The problem omicsTL solves

In many omics studies you have two kinds of data: a *source* — existing experiments
from related biological contexts (different strains, cohorts, tissue types, or
experimental conditions) — and a *target* — a smaller dataset for the specific
prediction task you care about.  A model trained only on the target often
performs poorly: too few samples relative to the number of measured features.
Discarding the source data wastes hard-won experimental knowledge.

omicsTL applies *transfer learning* to bridge the two.  It pre-trains a model on
the source data to learn which features carry signal, then fine-tunes it on the
target data to adapt to the new distribution.  The benefit is largest when the
target is small and the source and target share underlying biology.

## Running example

Throughout this vignette we classify samples as viral or bacterial infection using
proteomics data (475 proteins).  We have two cohorts:

| Cohort | Samples | Class balance | Role |
|---|---|---|---|
| Source | 358 | 179 bacterial / 179 viral | Pre-training |
| Target train | 192 | 45 bacterial / 147 viral | Fine-tuning |
| Target test | 48 | 15 bacterial / 33 viral | Evaluation only |

The target cohort is both small and imbalanced — a model trained on it alone
collapses to always predicting the majority class, yielding **31% accuracy**.
With source pre-training that jumps to **94%**.

---

## Installation

**Docker (recommended)** — bundles Python 3.12, R ≥ 4.2.0, and all dependencies:
```bash
git clone <repo-url> && cd timed-hpc
docker build . -t omicstl
docker run -it -v $(pwd):/workspace omicstl /bin/bash
pip install -e .
```
**VS Code Dev Container** — open the repo and select *Dev Containers: Reopen in Container*.

## Quick start

```python
import pandas as pd, torch, random
from torch import device
from omicstl.simulation_utils.data_utils import DatasetContainer
from omicstl.simulation_utils.model_utils import fit_dl_model
from omicstl import TIMEDClassifierMLP

datasets = DatasetContainer(
    source_data = pd.read_csv("source.csv").set_index("SampleID"),
    target_data = pd.read_csv("target_train.csv").set_index("SampleID"),
    target_test_data = [pd.read_csv("target_test.csv").set_index("SampleID")],
)
datasets.set_response_column("Resp")

random.seed(42); torch.manual_seed(42)
_, model, _ = fit_dl_model(datasets, "mult_mlp", device("cpu"), param_grid)

TIMEDClassifierMLP(model, classes=[1, 2]).save("mlp_model.pth")

clf    = TIMEDClassifierMLP.load("mlp_model.pth")
labels = clf.predict(new_data)   # → [2, 1, 2, 2, ...]
```

The rest of this vignette walks through each step and explains the choices.

In [ ]:
import warnings
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import device

warnings.filterwarnings("ignore")

---
## Step 1 — Load and prepare the data

omicsTL expects every DataFrame to be **numeric and complete** before it is passed
in — see the [Notes](#notes) section below for data requirements.
Here the response column arrives as the strings `"viral"` and `"bacterial"`,
which we encode to integers.  The dtype signals the task to the package:
`int64` triggers classification; `float64` (or any non-integer type)
triggers regression.

In [ ]:
data_dir = Path("data")

source_data  = pd.read_csv(data_dir / "source_dset.csv").set_index("SampleID")
target_train = pd.read_csv(data_dir / "target_transfer.csv").set_index("SampleID")
target_test  = pd.read_csv(data_dir / "target_validation.csv").set_index("SampleID")

label_map = {"viral": 2, "bacterial": 1}
for df in (source_data, target_train, target_test):
    if df["Resp"].dtype == object:
        df["Resp"] = df["Resp"].map(label_map).astype(np.int64)

for name, df in {"source": source_data, "target_train": target_train, "target_test": target_test}.items():
    counts = df["Resp"].value_counts().sort_index()
    print(f"{name:14s}: {df.shape[0]:>3} samples, {df.shape[1]-1} features"
          f"  |  bacterial={counts.get(1,0)}, viral={counts.get(2,0)}")

```
source        : 358 samples, 475 features  |  bacterial=179, viral=179
target_train  : 192 samples, 475 features  |  bacterial=45,  viral=147
target_test   :  48 samples, 475 features  |  bacterial=15,  viral=33
```

---
## Step 2 — Package data into a `DatasetContainer`

`DatasetContainer` is the single object consumed by all fitting functions.
It holds three core partitions:

| Argument | Role |
|---|---|
| `source_data` | Full source cohort — used for pre-training |
| `target_data` | Target adaptation set — used for fine-tuning |
| `target_test_data` | Held-out evaluation set(s) — never seen during training |

Pass `target_test_data` as a **list** even with a single test set.
An optional fourth argument, `target_ensemble_data`, is a small held-out target
partition used by the random forest to weight its transfer variants.  If omitted,
the RF fitter carves it out of `target_data` automatically; deep learning models
ignore it entirely.

After construction, call `set_response_column()` with the name of the response column.

In [ ]:
from omicstl.simulation_utils.data_utils import DatasetContainer

datasets = DatasetContainer(
    source_data     = source_data,
    target_data     = target_train,
    target_test_data = [target_test],   # list — required
)
datasets.set_response_column("Resp")

print(datasets)
print(f"\nTask type: {'classification' if datasets.is_classification() else 'regression'}")

```
DatasetContainer with:
  Source data: (358, 476) (not split)
  Target data: (192, 476)
  Target test data: 1 datasets

Task type: classification
```

---
## Step 3 — Fit the models

### How `fit_dl_model` works

```
fit_dl_model(data_container, model_type, torch_device, param_grid)

  1. Cross-validate on source data  →  select best hyperparameters
  2. Pre-train on all source data
  3. Copy weights  →  fine-tune on target training data    (tl_model)
  4. Train from scratch on target data only                (baseline_model)
  5. Evaluate both on target_test_data  →  results DataFrame

returns: (results, tl_model, baseline_model)
```

`model_type` is `"mult_mlp"` or `"mult_vae"`.
Swap `device("cpu")` for `device("cuda")` to use a GPU.

### Hyperparameter grid

`fit_dl_model` searches the Cartesian product of the values below using
cross-validation on the source data.  The parameters most worth varying for a new
dataset:

- **`dropout` / `weight_decay`** — regularisation; increase when the target
  is small or the feature count is high relative to sample size
- **`lr`** — learning rate; 0.01 is a good default for large source sets,
  0.001 for smaller ones
- **`gamma`** — focal-loss exponent; values above 1 concentrate learning on
  hard-to-classify samples, which helps with class imbalance
- **`source_epochs` / `target_epochs`** — upper bounds only; early stopping
  halts training when validation performance plateaus, so these rarely need tuning
- **`freeze`** — `"none"` keeps all weights trainable during fine-tuning;
  `"marginal"` freezes the feature-extraction layers and only updates the
  prediction head

In [ ]:
param_grid = {
    "dropout"        : [0.25, 0.5],
    "n_latent_dims"  : [2],
    "hidden_dim_base": [6],
    "lr"             : [0.01, 0.001],
    "source_epochs"  : [1000],
    "target_epochs"  : [1000],
    "freeze"         : ["none"],
    "weight_decay"   : [1e-4, 1e-2],
    "gamma"          : [1, 2, 3],
}

In [ ]:
from omicstl.simulation_utils.model_utils import fit_dl_model, fit_rf_model

random.seed(1123); torch.manual_seed(42)
mlp_results, mlp_model, _ = fit_dl_model(datasets, "mult_mlp", device("cpu"), param_grid)
print("MLP fit complete")

```
MLP fit complete
```

In [ ]:
torch.manual_seed(42)
vae_results, vae_model, _ = fit_dl_model(datasets, "mult_vae", device("cpu"), param_grid)
print("VAE fit complete")

```
VAE fit complete
```

In [ ]:
from omicstl.r_utils import set_seed

# set_seed() seeds the R random-number generator — required for RF reproducibility
random.seed(42); set_seed(42)
rf_results, rf_model = fit_rf_model(datasets)
print("RF fit complete")

```
RF fit complete
```

---
## Step 4 — Interpret the results

### Deep learning results table

Every DL results DataFrame has two rows:

| `model_id` | Description |
|---|---|
| `target` | Transfer-learning model — source pre-trained, then fine-tuned on target |
| `target_nosource` | Ablation baseline — same architecture, trained on target data only |

The gap between them measures the transfer learning benefit.  If `target` does not
outperform `target_nosource`, the source domain is likely too different from the
target, or the target dataset is already large enough to train from scratch.

### RF results table

The RF returns one row per transfer variant.  `pred_source_full` is the no-transfer
baseline (source RF applied directly to target data).  `pred_N_full` are the four
transfer algorithms; `pred_ensemble_full` is their weighted combination.
Compare each against `pred_source_full` to measure individual transfer benefit.

### Metrics

Classification: `acc`, `f1`, `mcc`, `roc_auc`, `precision`, `recall` — all higher
is better.  Regression: `rmse`, `mae` — lower is better.

In [ ]:
cols = ["model_id", "acc", "f1", "mcc", "roc_auc"]

print("MLP")
display(mlp_results[cols])

print("\nVAE")
display(vae_results[cols])

print("\nRF — source-only baseline vs best transfer variant vs ensemble")
rf_key = rf_results[rf_results["model_type"].isin(
    ["pred_source_full", "pred_3_full", "pred_ensemble_full"]
)][["model_type", "acc", "f1", "mcc", "roc_auc"]].reset_index(drop=True)
display(rf_key)

```
MLP
        model_id     acc        f1       mcc  roc_auc
0         target  0.9375  0.955224  0.858945      NaN
1  target_nosource  0.3125  0.000000  0.000000      NaN

VAE
        model_id     acc    f1       mcc  roc_auc
0         target  0.3125   0.0  0.000000      NaN
1  target_nosource  0.3125   0.0  0.000000      NaN

RF — source-only baseline vs best transfer variant vs ensemble
          model_type       acc        f1       mcc   roc_auc
0   pred_source_full  0.312500  0.000000  0.000000  0.363636
1       pred_3_full   0.645833  0.690909  0.349552  0.792929
2  pred_ensemble_full  0.416667  0.481481 -0.130243  0.787879
```

The MLP transfer model delivers a dramatic gain over its target-only ablation
(93.8% vs 31.3%).  The VAE matches its baseline — for single-view data
the VAE offers no advantage over the MLP (see [Appendix B](#appendix-b)).
The RF's best transfer variant (m4) reaches 64.6%, well above the source-only
baseline of 31.3%, though behind the MLP here.

In [ ]:
Path("model_outputs").mkdir(exist_ok=True)

labels = ["MLP\n(transfer)", "MLP\n(target-only)", "RF m4\n(transfer)", "RF\n(source-only)"]
accs   = [0.938,              0.313,                0.646,               0.313]
colors = ["#1565C0",          "#90CAF9",            "#2E7D32",           "#A5D6A7"]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(labels, accs, color=colors, width=0.55, edgecolor="white", linewidth=1.2)
ax.axhline(0.5, color="grey", linestyle="--", linewidth=0.9, label="50 % (random binary)")
ax.set_ylim(0, 1.1)
ax.set_ylabel("Accuracy on 48 held-out samples")
ax.set_title("Transfer learning vs baselines — viral / bacterial proteomics")
ax.legend(fontsize=9)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2, acc + 0.03,
            f"{acc:.0%}", ha="center", va="bottom", fontweight="bold")
plt.tight_layout()
plt.savefig("model_outputs/comparison.png", dpi=150)
plt.show()

---
## Step 5 — Save and deploy

`TIMEDClassifierMLP` and `TIMEDClassifierRF` wrap a trained model with a minimal
`save` / `load` / `predict` interface for external applications such as BacterAI.

For **classification**, pass `classes` — the sorted list of original response
values.  This maps the model's 0-indexed output back to your original labels
(e.g. `classes=[1, 2]` means output index 0 → label 1, index 1 → label 2).
For **regression**, omit `classes`.  The RF wrapper infers task type automatically.

In [ ]:
from omicstl import TIMEDClassifierMLP, TIMEDClassifierRF

out_dir          = Path("model_outputs")
response_classes = sorted(target_test["Resp"].unique())   # [1, 2]

mlp_clf = TIMEDClassifierMLP(mlp_model, classes=response_classes)
mlp_clf.save(str(out_dir / "mlp_model.pth"))

rf_clf = TIMEDClassifierRF(rf_model)
rf_clf.save(str(out_dir / "rf_model.pkl"))

print(f"Saved → {out_dir / 'mlp_model.pth'}")
print(f"Saved → {out_dir / 'rf_model.pkl'}")

```
Saved → model_outputs/mlp_model.pth
Saved → model_outputs/rf_model.pkl
```

In [ ]:
test_features = target_test.drop(columns=["Resp"])

mlp_loaded = TIMEDClassifierMLP.load(str(out_dir / "mlp_model.pth"))
rf_loaded  = TIMEDClassifierRF.load(str(out_dir / "rf_model.pkl"))

pred_df = pd.DataFrame({
    "True Label"   : target_test["Resp"].values,
    "MLP Predicted": mlp_loaded.predict(test_features),
    "RF Predicted" : rf_loaded.predict(test_features),
}, index=test_features.index)
pred_df["MLP Correct"] = pred_df["MLP Predicted"] == pred_df["True Label"]
pred_df["RF Correct"]  = pred_df["RF Predicted"]  == pred_df["True Label"]

display(pred_df.head(10))
print(f"\nMLP accuracy: {pred_df['MLP Correct'].mean():.1%}")
print(f"RF  accuracy: {pred_df['RF Correct'].mean():.1%}")

```
                        True Label  MLP Predicted  RF Predicted  MLP Correct  RF Correct
SampleID
Uni_MCK_D3_12_Hr_R5             1              1             2         True       False
Uni_CoV2_IT_D1_12_Hr_R3         2              2             2         True        True
Uni_NL63_D3_12_Hr_R3            2              2             1         True       False
Uni_MCK_D3_12_Hr_R2             1              1             2         True       False
Uni_NL63_D3_12_Hr_R5            2              2             2         True        True
...

MLP accuracy: 93.8%
RF  accuracy: 54.2%
```

In [ ]:
# ── Minimal integration snippet for BacterAI or any downstream application ───
#
# from omicstl import TIMEDClassifierMLP, TIMEDClassifierRF
# import pandas as pd
#
# mlp_clf = TIMEDClassifierMLP.load("mlp_model.pth")   # load once at startup
# rf_clf  = TIMEDClassifierRF.load("rf_model.pkl")
#
# new_data = pd.read_csv("new_samples.csv").set_index("SampleID")
# new_data = new_data[feature_names]  # align columns to training order — required
#
# mlp_labels = mlp_clf.predict(new_data)   # → [1, 2, 2, 1, ...]  (original label space)
# rf_labels  = rf_clf.predict(new_data)
print("See commented code above.")

---
## Which model family should you use?

**Start with `mult_mlp`.**  It is the fastest to train, easiest to tune, and most
stable across dataset sizes.  If the `target` row outperforms `target_nosource` in
the results table, the MLP is often sufficient.

**Try `mult_vae` when you have multiple separate omics layers.**  The VAE uses a
Product-of-Experts fusion in latent space that can capture shared biological signal
across modalities.  For a single feature matrix it offers no advantage over the MLP
and trains more slowly.  See [Appendix B](#appendix-b) for the multi-view workflow.

**Use `rf` when interpretability or compute constraints matter.**  The RF reports
per-feature importances from each transfer variant, making it easier to explain
which features drive the classification.  It also runs without a GPU and integrates
naturally with R-based downstream analyses.

<a id="notes"></a>

---
## Notes

### Data requirements

- **Source and target must have the same features.**  Every DataFrame passed to
  `DatasetContainer` must contain identical columns in the same order.
  omicsTL does not align or subset columns automatically.  If your source and target
  come from different platforms or experiments, harmonise the feature set before
  fitting (e.g. take the intersection of measured proteins and reorder).

- **Missing values must be imputed before fitting.**  omicsTL does not handle `NaN`
  internally.  Apply a suitable imputation strategy first — common choices for omics
  data include median/mean imputation per feature, k-NN imputation, or minimum-value
  imputation for data that are missing-not-at-random (e.g. below-detection proteins).

- **Categorical predictors must be numerically encoded.**  omicsTL expects a
  fully numeric feature matrix.  Convert any categorical covariates (batch label,
  sex, treatment group, tissue type) to numbers before passing them in — for example
  using one-hot encoding or ordinal encoding from `sklearn.preprocessing`.

### Response column

- The dtype of the response column determines the task.  Use `int64` for
  classification and `float64` for regression.  A column of `1.0` and `2.0` stored
  as `float64` will trigger regression, not binary classification — cast explicitly
  with `.astype(np.int64)` when needed.
- For classification, label values can be any integers (e.g. 1 and 2, or 0 and 1),
  but they must be positive and consistent across source and target.  Pass the sorted
  unique values as `classes` when wrapping with `TIMEDClassifierMLP` so predictions
  are mapped back to the original label space.

### Partitioning

- `target_test_data` must be a Python **list** even with a single test set.
- For classification, **stratify all target splits** so that every class appears in
  every partition.  If the ensemble or training partition is missing a class, the RF
  model will raise an error from R.
- `target_ensemble_data` is only consumed by the RF (5–10 samples is sufficient);
  it is ignored by `fit_dl_model`.

### Reproducibility

- Set `random.seed()` and `torch.manual_seed()` immediately before each
  `fit_dl_model` call.  `torch.manual_seed` does not seed R.
- For RF results, also call `set_seed()` from `omicstl.r_utils` immediately
  before `fit_rf_model`.

---
<a id="appendix-a"></a>

# Appendix A — Simulated Data

The simulation utilities are intended for **method development and benchmarking**:
they generate controlled source/target dataset pairs with known properties before
you apply the pipeline to real data.  If you already have source and target
datasets, skip this appendix entirely.

### How it works

Two real feature matrices serve as *covariance templates*.  The generator
preserves their correlation structure, adds Gaussian noise at a chosen
signal-to-noise ratio, and applies a user-defined response function.  Passing
the `lc_info` object returned by the **source** call back in as `prior_lc_info`
to the **target** call *couples* the two datasets — they share the same latent
structure, which is the precondition for transfer learning to be beneficial.
Without it, source and target are statistically independent.

```
response_function(func_str)
  func_str : R expression string.  Feature matrix is available as `df`
             (1-based column index: df[, 1] is the first column).
  returns  : R function object passed to generate_synth_data

generate_synth_data(
    data,                 # pd.DataFrame — covariance template (real feature matrix)
    num_features,         # int — number of output features
    num_samples,          # int — number of output samples
    response_fn,          # R function from response_function()
    response_parameters,  # None → continuous (float64)
                          # {"ncats": K, "quantile": "quantile"} → K-class int64
    snr,                  # float — signal-to-noise ratio; 1 = equal parts signal/noise
    prior_lc_info,        # pd.DataFrame | None — from the source call; couples target to source
)
returns: (synth_df, lc_info, cut_points)
  synth_df  : response in column 0, features in remaining columns
  lc_info   : pass as prior_lc_info to the paired target call
```

After generation: rename column 0 to your response name, cast to `int64` for
classification, split into train/ensemble/test partitions, and build a
`DatasetContainer` exactly as in the main workflow.

In [ ]:
from sklearn.model_selection import train_test_split
from omicstl.simulation_utils.data_generation import generate_synth_data, response_function

# Real feature matrices used only as covariance templates
source_real = pd.read_csv("data/source_data_real.csv", index_col=0)
target_real = pd.read_csv("data/target_data_real.csv", index_col=0)

response_fn = response_function("tanh(df[, 2]) + df[, 1] * df[, ncol(df)] ^ 2")

N_SOURCE, N_TRAIN, N_ENS, N_TEST = 100, 50, 5, 25

# Generate coupled source and target datasets
sim_source, lc_info, _ = generate_synth_data(
    data=source_real, num_features=100, num_samples=N_SOURCE,
    response_fn=response_fn, snr=1,
)
sim_target, _, _ = generate_synth_data(
    data=target_real, num_features=100, num_samples=N_TRAIN + N_ENS + N_TEST,
    response_fn=response_fn, prior_lc_info=lc_info,   # couples target to source
    snr=1,
)

# Rename response column (column 0); cast to int64 for classification
for df in (sim_source, sim_target):
    df.rename(columns={df.columns[0]: "response"}, inplace=True)
# df["response"] = df["response"].astype(np.int64)  # uncomment for categorical

# Split target into train / ensemble / test
# For classification add: stratify=sim_target["response"]
combined, test = train_test_split(sim_target, test_size=N_TEST, random_state=42)
train,    ens  = train_test_split(combined,   test_size=N_ENS,  random_state=42)

# Package and fit — identical API to the main workflow
sim_datasets = DatasetContainer(
    source_data=sim_source, target_data=train,
    target_ensemble_data=ens, target_test_data=[test],
)
sim_datasets.set_response_column("response")

random.seed(42); torch.manual_seed(42)
sim_results, _, _ = fit_dl_model(sim_datasets, "mult_mlp", device("cpu"), param_grid)
display(sim_results[["model_id", "rmse", "mae"]])

---
<a id="appendix-b"></a>

# Appendix B — Multi-view (Multi-omics) Usage

The omicsTL model architectures — `JointMLP`, `JointVAE`, and `TransferForest` —
were designed to accept **multiple separate omics matrices as views**.  Each view
gets its own marginal network (a small MLP or VAE encoder), and the view-level
representations are fused before the prediction head:

- **MLP** concatenates (or averages) the per-view hidden representations.
- **VAE** applies a Product-of-Experts fusion in latent space, which can extract
  shared biological signal across modalities.
- **RF** trains one forest per view, then transfers and ensembles across views.

The high-level API used in the main workflow (`DatasetContainer` +
`fit_dl_model` / `fit_rf_model`) treats all features as **a single view**.  To
use multiple separate omics layers you work directly with the lower-level classes
`TransferMLP`, `TransferVAE`, or `TransferForest`.

The sketch below shows the multi-view calling convention for `TransferMLP`.
Each function that accepts `views` takes a **list of DataFrames**, one per omics
type; the list length must be consistent across source, target, and test calls.

In [ ]:
# ── Multi-view sketch (not executable without multi-omics data) ───────────────
#
# from omicstl.transfer_networks import TransferMLP
# from omicstl.deep_learning_utils import PredictionMode
#
# Suppose you have two separate omics matrices per sample:
#   proteomics_source  (n_source × p_proteins)
#   transcriptomics_source  (n_source × g_genes)
# and matching target matrices.
#
# view_dims is the number of features in each view:
# view_dims = [p_proteins, g_genes]
#
# model = TransferMLP(
#     hidden_sizes = [[64, 32], [64, 32]],  # one list of layer sizes per view
#     dropout      = 0.25,
#     hidden_dim   = 64,
#     combine_fn   = "concat",
# )
# model.with_classification()   # or .with_regression()
# model.set_model_dims(view_dims=view_dims, output_dim=n_classes)
#
# # Pre-train on source
# model.create_model("source", lr=0.01, weight_decay=1e-4)
# model.source.train(
#     train_views  = [proteomics_source_train, transcriptomics_source_train],
#     y_train      = y_source_train,
#     loss_fn      = model._compute_loss,
#     epochs       = 1000,
#     test_views   = [proteomics_source_val, transcriptomics_source_val],
#     y_test       = y_source_val,
# )
#
# # Fine-tune on target (copies source weights)
# model.create_model("target", lr=0.01, weight_decay=1e-4, source_model="source")
# model.target.train(
#     train_views = [proteomics_target_train, transcriptomics_target_train],
#     y_train     = y_target_train,
#     loss_fn     = model._compute_loss,
#     epochs      = 1000,
# )
#
# # Predict
# preds = model.target.predict(
#     views = [proteomics_test, transcriptomics_test]
# )
print("See commented code above for the multi-view API sketch.")